# TRIDENT Output Data Validation
Validation of TRIDENT output. Inspects HDF5 structure (coords, features) for one slide, then sweeps all .h5 files to confirm consistent feature dimensionality, encoder, and magnification, plus per-slide patch-count distribution and NaN/Inf sanity checks.

## Imports | Paths

In [11]:
import os
import glob
import argparse
import h5py as h5py
import numpy as np
from pathlib import Path

In [12]:
PROJECT_ROOT = Path().resolve().parent
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "features_uni_v1_sample30"
SINGLE_OUTPUT = PROCESSED_PATH / "TCGA-05-4244-01Z-00-DX1.d4ff32cd-38cf-40ea-8213-45c2b100ac01.h5"

## Inspect HDF5 Structure

### Helper Function

In [13]:
def inspect_one(path, full=False):
    with h5py.File(path, "r") as h:
        feats = h["features"]
        coords = h["coords"]
        n, dim = feats.shape
        a = dict(coords.attrs)
        row = {
            "name": os.path.basename(path),
            "n_patches": n,
            "dim": dim,
            "dtype": str(feats.dtype),
            "encoder": h["features"].attrs.get("encoder", "?"),
            "mag": a.get("target_magnification", "?"),
            "patch": a.get("patch_size", "?"),
        }

        if full:
            x = feats[:]
            row["nan"] = int(np.isnan(x).sum())
            row["inf"] = int(np.isinf(x).sum())
            row["min"], row["max"] = float(x.min()), float(x.max())
            row["mean"], row["std"] = float(x.mean()), float(x.std())
            print(f"{row['name']}")
            print(f"\tfeatures\t: {n} x {dim}  {row['dtype']}  ({row['encoder']}, {row['mag']}x, patch {row['patch']})")
            print(f"\tcoords\t\t: x[{coords[:,0].min()}..{coords[:,0].max()}] y[{coords[:,1].min()}..{coords[:,1].max()}]")
            print(f"\tvalues\t\t: min {row['min']:.3f}  max {row['max']:.3f}  mean {row['mean']:.3f}  std {row['std']:.3f}")
            print(f"\tbad\t\t\t: NaN {row['nan']}  Inf {row['inf']}")

        return row

In [14]:
with h5py.File(SINGLE_OUTPUT, 'r') as h5:
    print(list(h5.keys()))
    print(f"\t{h5['coords']}")
    print(f"\t{h5['features']}")
    print()
    
inspect_one(SINGLE_OUTPUT, full=True)

['coords', 'features']
	<HDF5 dataset "coords": shape (2975, 2), type "<i8">
	<HDF5 dataset "features": shape (2975, 1024), type "<f4">

TCGA-05-4244-01Z-00-DX1.d4ff32cd-38cf-40ea-8213-45c2b100ac01.h5
	features	: 2975 x 1024  float32  (uni_v1, 20x, patch 256)
	coords		: x[4096..41984] y[4608..38912]
	values		: min -6.530  max 7.563  mean -0.013  std 1.213
	bad			: NaN 0  Inf 0


{'name': 'TCGA-05-4244-01Z-00-DX1.d4ff32cd-38cf-40ea-8213-45c2b100ac01.h5',
 'n_patches': 2975,
 'dim': 1024,
 'dtype': 'float32',
 'encoder': 'uni_v1',
 'mag': np.int64(20),
 'patch': np.int64(256),
 'nan': 0,
 'inf': 0,
 'min': -6.530282020568848,
 'max': 7.562708854675293,
 'mean': -0.013456428423523903,
 'std': 1.2129400968551636}

## Validate All Slide H5 Files

In [22]:
if os.path.isdir(PROCESSED_PATH):
    files = sorted(glob.glob(os.path.join(PROCESSED_PATH, "*.h5")))
else:
    files = [PROCESSED_PATH]
if not files:
    raise SystemExit(f"no .h5 files under {PROCESSED_PATH}")

rows = [inspect_one(f, len(files) == 1) for f in files]


counts = np.array([r["n_patches"] for r in rows])
dims = {r["dim"] for r in rows}
encoders = {r["encoder"] for r in rows}
mags = {str(r["mag"]) for r in rows}

print(f"\n{'File':<52}{'Patches':>10}{'Dim':>7}")

for r in rows:
    print(f"{r['name'][:50]:<52}{r['n_patches']:>10}{r['dim']:>7}")
print()

print(f"{len(files)} slides | Total Patches {counts.sum():,} | Min {counts.min()} / Median {int(np.median(counts))} / Max {counts.max()}")
print()
print(f"Feature Dim(s): {dims} \t Encoder(s): {encoders} \t Magnification(s): {mags}")
if len(dims) > 1 or len(encoders) > 1:
    print("WARNING: Inconsistent dim/encoder across slides")


File                                                   Patches    Dim
TCGA-05-4244-01Z-00-DX1.d4ff32cd-38cf-40ea-8213-45        2975   1024
TCGA-05-4245-01Z-00-DX1.36ff5403-d4bb-4415-b2c5-7c        2032   1024
TCGA-05-4249-01Z-00-DX1.9fce0297-cc19-4c04-872c-44        5075   1024
TCGA-05-4250-01Z-00-DX1.90f67fdf-dff9-46ca-af71-09        3965   1024
TCGA-05-4382-01Z-00-DX1.76b49a4c-dbbb-48b0-b677-6d        3274   1024
TCGA-05-4395-01Z-00-DX1.20205276-ca16-46b2-914a-fe        2568   1024
TCGA-05-4396-01Z-00-DX1.49DD5F68-7473-4945-B384-EA        5908   1024
TCGA-05-4397-01Z-00-DX1.00e9cdb3-b50e-439c-86b0-d7        6301   1024
TCGA-05-4398-01Z-00-DX1.269bc75f-492e-48b1-87ee-85        8529   1024
TCGA-05-4402-01Z-00-DX1.c653ddc2-88c1-45ac-88e7-4e        7589   1024
TCGA-05-4403-01Z-00-DX1.fee8e988-956c-42a2-a6c5-06        5316   1024
TCGA-05-4405-01Z-00-DX1.D57EC2B2-3A59-4954-86A7-61        3101   1024
TCGA-05-4415-01Z-00-DX1.55E0C429-B308-4962-8DA9-41        3858   1024
TCGA-05-4417-01Z-00